In [ ]:
# --- الخلية 1: التثبيت وإعادة التشغيل ---
!pip install "numpy<2" matplotlib opencv-python scipy tqdm albumentations segmentation-models-pytorch pandas scikit-learn scikit-image pyarrow fastparquet wfdb
!pip install ultralytics
import os
import sys

if 'google.colab' in sys.modules:
    print("⏳ جاري تثبيت المكتبات... سيتم إعادة التشغيل.")
    os.kill(os.getpid(), 9)

In [17]:
# --- الخلية 2: الاستيرادات المحدثة (Updated Imports) ---
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import random
from tqdm import tqdm
import albumentations as A
import torch
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
import glob
from ultralytics import YOLO
import shutil
import warnings
import wfdb
import io
import torch.nn as nn
import gc
# --- [جديد] مكتبات الهندسة المتقدمة ---
from skimage.transform import ProjectiveTransform, warp
from skimage.measure import ransac
from scipy.fft import fft2, fftshift
from torch.optim.lr_scheduler import CosineAnnealingLR
# إعدادات العرض والتنبيهات
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"✅ تم تحديث المكتبات: جاهزون للمعالجة الهندسية والفيزيائية.")
print(f"🚀 الجهاز المستخدم: {DEVICE}")

✅ تم تحديث المكتبات: جاهزون للمعالجة الهندسية والفيزيائية.
🚀 الجهاز المستخدم: cuda


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [5]:
# --- الخلية 4: الضبط الدقيق على الواقع (Phase 10: Real World Fine-Tuning) - [M10: PSEUDO] ---

# تنظيف الذاكرة
gc.collect()
torch.cuda.empty_cache()
plt.ioff()

# إعداد المسارات الحقيقية
REAL_IMG_DIR = "/kaggle/input/physionet-ecg-image-digitization"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. تحميل الموديل المعلم (Phase 9 Teacher)
print("👨‍🏫 تحميل المعلم (Phase 9 Model) لتوليد التسميات الزائفة (Pseudo-Labels)...")
teacher_model = smp.Unet(
    encoder_name="resnet50", 
    encoder_weights=None, 
    in_channels=3, 
    classes=1, 
    decoder_attention_type="scse"
)
if os.path.exists("best_model_phase9.pth"):
    teacher_model.load_state_dict(torch.load("best_model_phase9.pth"))
else:
    print("⚠️ تحذير: موديل المرحلة 9 غير موجود، نستخدم تهيئة عشوائية (غير مستحسن).")

teacher_model.to(DEVICE)
teacher_model.eval()

# 2. تجهيز مجموعة البيانات الحقيقية
class RealWorldDataset(Dataset):
    def __init__(self, img_dir, model, transform=None):
        self.image_paths = glob.glob(f"{img_dir}/**/*.png", recursive=True) + \
                           glob.glob(f"{img_dir}/**/*.jpg", recursive=True)
        # نستخدم عينة 977 صورة (أو كل ما هو موجود)
        self.image_paths = self.image_paths[:1500] 
        self.model = model
        self.transform = transform
        self.pseudo_masks = {} # كاش للأقنعة لتسريع التدريب
        
        print(f"🔄 جاري توليد الأقنعة لـ {len(self.image_paths)} صورة حقيقية...")
        self._generate_pseudo_labels()
        
    def _generate_pseudo_labels(self):
        # توليد الأقنعة مرة واحدة وحفظها في الرام
        with torch.no_grad():
            for path in tqdm(self.image_paths, desc="Generating Pseudo-Labels"):
                try:
                    img = cv2.imread(path)
                    if img is None: continue
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img_resized = cv2.resize(img, (512, 256))
                    
                    # التنبؤ
                    tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float().unsqueeze(0).to(DEVICE) / 255.0
                    pred = self.model(tensor)
                    mask = torch.sigmoid(pred).cpu().numpy()[0][0]
                    
                    # تنظيف القناع (Thresholding) ليكون ثنائياً وحاداً
                    mask = (mask > 0.5).astype(np.float32)
                    
                    self.pseudo_masks[path] = mask
                except:
                    continue
                    
    def __len__(self):
        return len(self.pseudo_masks)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        if path not in self.pseudo_masks: 
            # Fallback random
            return torch.zeros((3, 256, 512)), torch.zeros((1, 256, 512))
            
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (512, 256))
        
        mask = self.pseudo_masks[path]
        
        # تطبيق تعزيزات خفيفة جداً (Flip/Rotate) لزيادة المتانة
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
            
        img = img.transpose(2, 0, 1).astype('float32') / 255.0
        mask = np.expand_dims(mask, axis=0)
        
        return torch.from_numpy(img), torch.from_numpy(mask)

# تعزيزات خفيفة للضبط الدقيق (لا نريد تشويهاً كبيراً، فقط تحسين التعميم)
fine_tune_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

# إنشاء الداتاسيت
real_dataset = RealWorldDataset(REAL_IMG_DIR, teacher_model, transform=fine_tune_aug)
real_dataloader = DataLoader(real_dataset, batch_size=16, shuffle=True, num_workers=0)

# 3. إعداد التدريب
# نعيد تحميل الموديل ليكون هو "الطالب" الذي سنتدرب عليه
student_model = smp.Unet(
    encoder_name="resnet50", 
    encoder_weights=None, 
    in_channels=3, 
    classes=1, 
    decoder_attention_type="scse"
)
student_model.load_state_dict(torch.load("best_model_phase9.pth")) # نبدأ من حيث انتهينا
student_model.to(DEVICE)

# الإعدادات المطلوبة
EPOCHS = 20
LR = 0.0001 # معدل منخفض كما طلبت

optimizer = torch.optim.AdamW(student_model.parameters(), lr=LR, weight_decay=1e-2)
loss_fn = smp.losses.DiceLoss(mode='binary') # نركز على الشكل فقط الآن

print("🔥 بدء الضبط الدقيق على الواقع (Phase 10: Real Fine-Tuning)...")
print(f"📊 المواصفات: {len(real_dataset)} Real Images | LR={LR} | 20 Epochs")

student_model.train()
best_loss = float('inf')

for epoch in range(EPOCHS):
    epoch_loss = 0
    loop = tqdm(real_dataloader, desc=f"Fine-Tuning {epoch+1}/{EPOCHS}", leave=False)
    
    for images, masks in loop:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        
        prediction = student_model(images)
        loss = loss_fn(prediction, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    
    avg_loss = epoch_loss / len(real_dataloader)
    print(f"Epoch {epoch+1}: Real Loss={avg_loss:.5f}")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(student_model.state_dict(), "best_model_phase10.pth")

print(f"✅ انتهى الضبط الدقيق! أفضل خسارة واقعية: {best_loss:.5f}")
print("💾 تم حفظ النموذج النهائي: best_model_phase10.pth")

👨‍🏫 تحميل المعلم (Phase 9 Model) لتوليد التسميات الزائفة (Pseudo-Labels)...
🔄 جاري توليد الأقنعة لـ 1500 صورة حقيقية...


Generating Pseudo-Labels: 100%|██████████| 1500/1500 [07:40<00:00,  3.26it/s]


🔥 بدء الضبط الدقيق على الواقع (Phase 10: Real Fine-Tuning)...
📊 المواصفات: 1500 Real Images | LR=0.0001 | 20 Epochs


Epoch 1: Real Loss=0.20827


Epoch 2: Real Loss=0.15751


...

Epoch 16: Real Loss=0.11156


Epoch 17: Real Loss=0.11037


Epoch 18: Real Loss=0.11144


Epoch 19: Real Loss=0.11090


Epoch 20: Real Loss=0.10952
✅ انتهى الضبط الدقيق! أفضل خسارة واقعية: 0.10952
💾 تم حفظ النموذج النهائي: best_model_phase10.pth


In [ ]:
# --الخلية 5 ---
def viterbi_path(prob_map, lam=10.0, max_jump=5):
    H, W = prob_map.shape
    cost_map = -np.log(prob_map + 1e-6)

    dp = np.full((H, W), np.inf)
    parent = np.zeros((H, W), dtype=int)

    dp[:, 0] = cost_map[:, 0]

    for x in range(1, W):
        for y in range(H):
            y_prev_min = max(0, y - max_jump)
            y_prev_max = min(H, y + max_jump + 1)
            
            prev_costs = dp[y_prev_min:y_prev_max, x-1]
            dist_penalty = lam * np.abs(np.arange(y_prev_min, y_prev_max) - y)
            
            total_costs = prev_costs + dist_penalty
            
            best_idx = np.argmin(total_costs)
            min_cost = total_costs[best_idx]
            
            dp[y, x] = cost_map[y, x] + min_cost
            parent[y, x] = y_prev_min + best_idx

    path = np.zeros(W, dtype=int)
    path[-1] = np.argmin(dp[:, -1])
    
    for x in range(W-2, -1, -1):
        path[x] = parent[path[x+1], x+1]
        
    return path

def extract_signal_viterbi(prob_map):
    H, W = prob_map.shape
    
    path_indices = viterbi_path(prob_map)
    
    refined_signal = []
    win = 2
    
    for x, y_int in enumerate(path_indices):
        y_start = max(0, y_int - win)
        y_end = min(H, y_int + win + 1)
        
        weights = prob_map[y_start:y_end, x]
        indices = np.arange(y_start, y_end)
        
        if np.sum(weights) > 1e-5:
            y_subpixel = np.sum(weights * indices) / np.sum(weights)
        else:
            y_subpixel = y_int
            
        refined_signal.append(H - y_subpixel)
        
    return np.array(refined_signal)

In [ ]:
# --- الخلية 6---
model.eval()
with torch.no_grad():
    sig = generator.generate_synthetic_heartbeat(length=2000)
    img, mask = generator.plot_signal_to_image(sig)
    aug_img, aug_mask = generator.augment_pair(img, mask)
    
    input_tensor = torch.from_numpy(aug_img).permute(2, 0, 1).float() / 255.0
    input_tensor = cv2.resize(input_tensor.permute(1, 2, 0).numpy(), (512, 256))
    input_tensor = torch.from_numpy(input_tensor).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    
    pred_raw = model(input_tensor)
    prob_map = torch.sigmoid(pred_raw).cpu().numpy()[0][0]
    
    extracted_signal = extract_signal_viterbi(prob_map)
    
    extracted_series = pd.Series(extracted_signal)
    extracted_signal = extracted_series.interpolate(method='linear', limit_direction='both').values

plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.title("1. Original Input")
plt.imshow(cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.subplot(3, 1, 2)
plt.title("2. AI Probability Map (Input to Viterbi)")
plt.imshow(prob_map, cmap='magma')
plt.axis('off')

plt.subplot(3, 1, 3)
plt.title("3. Viterbi Extracted Signal (Smoother & Connected)")
plt.plot(extracted_signal, color='blue', linewidth=1.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"✅ تم استخدام Viterbi بنجاح!")

In [ ]:
# --- الخلية 7 ---
def process_full_page(image_full, model, num_rows=4):
    h, w, c = image_full.shape
    row_height = h // num_rows 
    
    full_signals = []
    
    print(f"بدء معالجة الصفحة (الحجم: {w}x{h}) - تقسيمها إلى {num_rows} صفوف...")
    
    plt.figure(figsize=(15, 12))
    
    for i in range(num_rows):
        y_start = i * row_height
        y_end = (i + 1) * row_height
        strip = image_full[y_start:y_end, :]
        
        display_strip = strip.copy()
        
        input_strip = cv2.resize(strip, (512, 256))
        input_tensor = torch.from_numpy(input_strip).permute(2, 0, 1).float() / 255.0
        input_tensor = input_tensor.unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            pred_raw = model(input_tensor)
            prob_map = torch.sigmoid(pred_raw).cpu().numpy()[0][0]
        
        signal_1d = extract_signal_viterbi(prob_map)
        
        s_series = pd.Series(signal_1d)
        signal_clean = s_series.interpolate(method='linear', limit_direction='both').fillna(0).values
        
        full_signals.append(signal_clean)
        
        plt.subplot(num_rows, 2, (i*2)+1)
        plt.imshow(cv2.cvtColor(display_strip, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        if i == 0: plt.title("Original Strip")
        
        plt.subplot(num_rows, 2, (i*2)+2)
        plt.plot(signal_clean, color='blue', linewidth=1)
        plt.ylim(0, 256) 
        plt.grid(True, alpha=0.2)
        plt.axis('off')
        if i == 0: plt.title(f"Extracted Signal (Viterbi)")

    plt.tight_layout()
    plt.show()
    
    return full_signals

print("1️⃣ جاري توليد صفحة ECG كاملة وهمية...")
rows = []
for _ in range(4):
    sig = generator.generate_synthetic_heartbeat(length=2000)
    img, _ = generator.plot_signal_to_image(sig)
    aug_img, _ = generator.augment_pair(img, _) 
    rows.append(aug_img)

full_page_image = np.vstack(rows)

print("2️⃣ تشغيل نظام الرقمنة (Viterbi Engine)...")
model.eval()
signals = process_full_page(full_page_image, model, num_rows=4)

print(f"✅ تمت المهمة! حصلنا على {len(signals)} إشارات رقمية.")

In [ ]:
# --- الخلية 8:  ---

def create_submission_file(model, num_samples=5):
    submission_data = []
    
    print(f"📦 جاري تجهيز ملف التسليم لـ {num_samples} عينات (باستخدام Viterbi)...")
    
    for i in range(num_samples):
        sample_id = f"sample_{i+1}"
        
        full_signals = []
        for _ in range(12): 
            sig = generator.generate_synthetic_heartbeat(length=1000)
            img, _ = generator.plot_signal_to_image(sig)
            
            input_strip = cv2.resize(img, (512, 256))
            input_tensor = torch.from_numpy(input_strip).permute(2, 0, 1).float() / 255.0
            input_tensor = input_tensor.unsqueeze(0).to(DEVICE)
            
            with torch.no_grad():
                pred = model(input_tensor)
                prob_map = torch.sigmoid(pred).cpu().numpy()[0][0]
            
            signal_1d = extract_signal_viterbi(prob_map)
            
            s_series = pd.Series(signal_1d)
            signal_clean = s_series.interpolate(method='linear', limit_direction='both').fillna(0).values
            
            target_len = 1000 
            signal_resampled = resample(signal_clean, target_len)
            
            full_signals.append(signal_resampled)
        
        row_dict = {'id': sample_id}
        for lead_idx, signal_data in enumerate(full_signals):
            row_dict[f'lead_{lead_idx+1}'] = np.round(signal_data, 4)
            
        submission_data.append(row_dict)

    df = pd.DataFrame(submission_data)
    df.to_csv("submission_simulated.csv", index=False)
    return df

df_submission = create_submission_file(model, num_samples=3)
print("\n✅ تم إنشاء ملف التسليم بنجاح.")
print("معاينة البيانات:")
print(df_submission.iloc[0, 1][:30])

In [ ]:
# --- الخلية 9:  ---
INPUT_DIR = "/kaggle/input"
COMPETITION_NAME = "physionet-ecg-image-digitization"

dataset_path = None
for dirname in os.listdir(INPUT_DIR):
    if COMPETITION_NAME in dirname:
        dataset_path = os.path.join(INPUT_DIR, dirname)
        print(f"✅ تم العثور على مجلد البيانات في: {dataset_path}")
        break

if dataset_path:
    print("\n📂 محتويات المجلد الرئيسي:")
    files = os.listdir(dataset_path)
    print(files)

    csv_file = None
    for f in files:
        if f.endswith(".csv") and "train" in f:
            csv_file = os.path.join(dataset_path, f)
            break
            
    if csv_file:
        print(f"\n📄 جاري قراءة الملف: {csv_file}")
        df = pd.read_csv(csv_file)
        
        print(f"عدد العينات في الملف: {len(df)}")
        print("أول 5 صفوف من البيانات:")
        display(df.head())
        
        print("\n🔍 فحص مسارات الصور...")
        image_example = glob.glob(f"{dataset_path}/**/*.png", recursive=True)
        if not image_example:
            image_example = glob.glob(f"{dataset_path}/**/*.jpg", recursive=True)
            
        if image_example:
            print(f"✅ تم العثور على صور! مثال لمسار صورة:\n{image_example[0]}")
        else:
            print("⚠️ لم يتم العثور على صور بامتداد png/jpg مباشرة، قد تكون مضغوطة أو في مسار آخر.")
            
    else:
        print("❌ لم يتم العثور على ملف train.csv!")
else:
    print("❌ خطأ: لم يتم العثور على مجلد المسابقة داخل /kaggle/input")
    print("تأكد من ضغط زر '+ Add Input' وإضافة مسابقة 'PhysioNet ECG Image Digitization'.")

In [ ]:
# الخلية 10 
def test_robust_real_image(model, dataset_path):
    print("🕵️‍♂️ جاري البحث عن ملفات صور حقيقية داخل المجلدات...")
    
    search_pattern_png = os.path.join(dataset_path, "train", "*.png")
    search_pattern_jpg = os.path.join(dataset_path, "train", "*.jpg")
    
    all_images = glob.glob(search_pattern_png) + glob.glob(search_pattern_jpg)
    
    if not all_images:
        print("⚠️ مجلد train يبدو فارغاً أو الصور في مكان آخر، سأبحث في test...")
        all_images = glob.glob(os.path.join(dataset_path, "test", "*.png")) + \
                     glob.glob(os.path.join(dataset_path, "test", "*.jpg"))

    if not all_images:
        print("❌ خطأ: لم يتم العثور على أي صور في المجلدات!")
        print(f"المسار الذي بحثت فيه: {dataset_path}")
        try:
            print("محتويات المجلد الرئيسي:", os.listdir(dataset_path))
            if os.path.exists(os.path.join(dataset_path, "train")):
                 print("محتويات train:", os.listdir(os.path.join(dataset_path, "train"))[:5])
        except:
            pass
        return

    print(f"✅ تم العثور على {len(all_images)} صورة.")
    
    img_path = random.choice(all_images)
    image_name = os.path.basename(img_path)
    print(f"📸 تم اختيار الصورة: {image_name}")
    print(f"📂 المسار الكامل: {img_path}")

    real_img = cv2.imread(img_path)
    if real_img is None:
        print("❌ فشل قراءة ملف الصورة (قد يكون تالفاً).")
        return
        
    h, w, c = real_img.shape
    print(f"📏 الأبعاد: {w}x{h}")

    crop_h = 400 
    y_center = h // 2
    y_start = max(0, y_center - (crop_h // 2))
    y_end = min(h, y_center + (crop_h // 2))
    
    real_strip = real_img[y_start:y_end, :]
    
    input_strip = cv2.resize(real_strip, (512, 256))
    input_tensor = torch.from_numpy(input_strip).permute(2, 0, 1).float() / 255.0
    input_tensor = input_tensor.unsqueeze(0).to(DEVICE)
    
    model.eval()
    with torch.no_grad():
        pred_raw = model(input_tensor)
        prob_map = torch.sigmoid(pred_raw).cpu().numpy()[0][0]
    
    signal = extract_signal_viterbi(prob_map)
    
    plt.figure(figsize=(15, 12))
    
    plt.subplot(3, 1, 1)
    plt.title(f"1. Real Image: {image_name}")
    plt.imshow(cv2.cvtColor(real_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    
    plt.subplot(3, 1, 2)
    plt.title("2. AI Probability Heatmap (Bright areas = Signal)")
    plt.imshow(prob_map, cmap='magma')
    plt.axis('off')
    
    plt.subplot(3, 1, 3)
    plt.title("3. Extracted Signal (Viterbi)")
    plt.plot(signal, color='blue', linewidth=1.5)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

if 'dataset_path' in globals() and dataset_path:
    test_robust_real_image(model, dataset_path)
else:
    print("⚠️ متغير dataset_path غير معرف! تأكد من تشغيل الخلية 8 بنجاح.")

In [ ]:
# الخلية 11
def estimate_grid_size(image_path_or_array, display=True):
    if isinstance(image_path_or_array, str):
        img = cv2.imread(image_path_or_array)
    else:
        img = image_path_or_array
        
    if img is None: return None
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    row_sum = np.sum(edges, axis=1)
    col_sum = np.sum(edges, axis=0)
    
    peaks_x, _ = find_peaks(col_sum, distance=10, prominence=50)
    peaks_y, _ = find_peaks(row_sum, distance=10, prominence=50)
    
    if len(peaks_x) > 1:
        diffs_x = np.diff(peaks_x)
        grid_width = np.median(diffs_x)
    else:
        grid_width = 0
        
    if len(peaks_y) > 1:
        diffs_y = np.diff(peaks_y)
        grid_height = np.median(diffs_y)
    else:
        grid_height = 0
    
    final_grid_size = 0
    if grid_width > 0 and grid_height > 0:
        final_grid_size = (grid_width + grid_height) / 2
    elif grid_width > 0:
        final_grid_size = grid_width
    elif grid_height > 0:
        final_grid_size = grid_height
        
    if final_grid_size < 5: 
        print("⚠️ تحذير: الشبكة غير واضحة، سيتم استخدام قيمة افتراضية.")
        final_grid_size = 25.0

    if display:
        plt.figure(figsize=(10, 4))
        plt.plot(col_sum, color='green', alpha=0.6)
        plt.plot(peaks_x, col_sum[peaks_x], "x", color='red')
        plt.title(f"Grid Detection Analysis (Estimated Box Size: {final_grid_size:.2f} pixels)")
        plt.xlabel("Pixel Position")
        plt.ylabel("Edge Intensity")
        plt.show()
        
    return final_grid_size

def calibrate_signal(raw_signal, grid_size_pixels):
    pixels_per_mV = grid_size_pixels * 10
    
    baseline = np.median(raw_signal)
    signal_centered = raw_signal - baseline
    
    signal_mV = signal_centered / pixels_per_mV
    
    return signal_mV

print("✅ تم تفعيل نظام المعايرة (Grid Calibrator).")

In [ ]:
# خلية 12 
if 'dataset_path' not in globals() or not dataset_path:
    possible_roots = ["/kaggle/input/physionet-ecg-image-digitization", "/kaggle/input"]
    for r in possible_roots:
        if os.path.exists(r):
            dataset_path = r
            break

def get_any_image_path(root_path):
    patterns = [
        f"{root_path}/**/*.png",
        f"{root_path}/**/*.jpg",
        f"{root_path}/**/**/*.png"
    ]
    for p in patterns:
        files = glob.glob(p, recursive=True)
        if files:
            return files
    return []

print("🔍 جاري البحث عن صور في كل المجلدات...")
real_images = get_any_image_path(dataset_path)

if real_images:
    test_img_path = random.choice(real_images)
    print(f"🧪 تحليل الصورة: {os.path.basename(test_img_path)}")
    print(f"📂 المسار: {test_img_path}")
    
    original_img = cv2.imread(test_img_path)
    if original_img is None:
        print("❌ الملف تالف، حاول مرة أخرى.")
    else:
        h, w, c = original_img.shape
        crop_h = 400
        y_start = max(0, h//2 - crop_h//2)
        real_strip = original_img[y_start:y_start+crop_h, :]
        
        print("1️⃣ جاري حساب حجم الشبكة (Grid Detection)...")
        grid_size = estimate_grid_size(real_strip, display=True)
        print(f"📏 حجم المربع الصغير المقدر: {grid_size:.2f} بكسل")
        
        print("2️⃣ استخراج الإشارة...")
        input_strip = cv2.resize(real_strip, (512, 256))
        input_tensor = torch.from_numpy(input_strip).permute(2, 0, 1).float() / 255.0
        input_tensor = input_tensor.unsqueeze(0).to(DEVICE)
        
        model.eval()
        with torch.no_grad():
            pred_raw = model(input_tensor)
            prob_map = torch.sigmoid(pred_raw).cpu().numpy()[0][0]
        
        raw_signal = extract_signal_viterbi(prob_map)
        
        grid_size_model = estimate_grid_size(input_strip, display=False)
        
        if grid_size_model < 5: 
            print("⚠️ الشبكة في الصورة المصغرة غير واضحة، سنستخدم نسبة وتناسب من الصورة الأصلية.")
            scale_factor = 256 / crop_h
            grid_size_model = grid_size * scale_factor
            
        print(f"📏 حجم الشبكة المعدل (للمعايرة): {grid_size_model:.2f} بكسل")
        
        calibrated_signal = calibrate_signal(raw_signal, grid_size_model)
        
        plt.figure(figsize=(12, 8))
        
        plt.subplot(3, 1, 1)
        plt.title("Original Strip")
        plt.imshow(cv2.cvtColor(real_strip, cv2.COLOR_BGR2RGB))
        plt.axis('off')

        plt.subplot(3, 1, 2)
        plt.title("Before: Raw Signal (Pixel Heights)")
        plt.plot(raw_signal)
        plt.ylabel("Pixels")
        plt.grid(True, alpha=0.3)
        
        plt.subplot(3, 1, 3)
        plt.title("After: Calibrated Signal (mV)")
        plt.plot(calibrated_signal, color='red')
        plt.ylabel("Voltage (mV)")
        plt.grid(True, alpha=0.3)
        plt.axhline(0.5, color='green', linestyle='--', alpha=0.5, label='0.5 mV')
        plt.axhline(-0.5, color='green', linestyle='--', alpha=0.5)
        plt.legend()
        
        plt.tight_layout()
        plt.show()
        
        print("✅ النتيجة النهائية: انظر للرسم الأحمر، هل القيم منطقية؟")

else:
    print("❌ خطأ غريب: لم يتم العثور على أي صور حتى مع البحث الشامل!")
    print(f"المسار الذي نبحث فيه: {dataset_path}")
    print("تأكد أنك أضفت البيانات (Add Data) من القائمة الجانبية.") 

In [ ]:
# خلية 13
def predict_long_image(image, model, window_size=512, step=512):
    h, w, c = image.shape
    
    target_h = 256
    scale_ratio = target_h / h
    new_w = int(w * scale_ratio)
    
    resized_img = cv2.resize(image, (new_w, target_h))
    
    full_prob_map = np.zeros((target_h, new_w), dtype=np.float32)
    counts_map = np.zeros((target_h, new_w), dtype=np.float32)
    
    for x in range(0, new_w, step):
        x_end = min(x + window_size, new_w)
        width = x_end - x
        
        window = resized_img[:, x:x_end, :]
        
        if width < window_size:
            pad = np.zeros((target_h, window_size, 3), dtype=np.uint8)
            pad[:, :width, :] = window
            window = pad
            
        input_tensor = torch.from_numpy(window).permute(2, 0, 1).float() / 255.0
        input_tensor = input_tensor.unsqueeze(0).to(DEVICE)
        
        model.eval()
        with torch.no_grad():
            pred = model(input_tensor)
            prob = torch.sigmoid(pred).cpu().numpy()[0][0]
            
        if width < window_size:
            full_prob_map[:, x:x_end] += prob[:, :width]
            counts_map[:, x:x_end] += 1
        else:
            full_prob_map[:, x:x_end] += prob
            counts_map[:, x:x_end] += 1

    full_prob_map /= np.maximum(counts_map, 1)
    
    return full_prob_map, scale_ratio

print("🚀 تشغيل نظام Sliding Window على صورة حقيقية...")

real_images = get_any_image_path(dataset_path)

if real_images:
    test_img_path = random.choice(real_images)
    print(f"📄 الصورة: {os.path.basename(test_img_path)}")
    
    original_img = cv2.imread(test_img_path)
    
    h, w, _ = original_img.shape
    crop_h = 400
    y_start = max(0, h//2 - crop_h//2)
    real_strip = original_img[y_start:y_start+crop_h, :] 
    
    print("1️⃣ حساب الشبكة...")
    grid_size_original = estimate_grid_size(real_strip, display=False)
    
    print("2️⃣ بناء خريطة الاحتمالات الكاملة...")
    big_prob_map, scale_ratio = predict_long_image(real_strip, model)
    
    print("3️⃣ استخراج المسار (Viterbi)...")
    raw_signal = extract_signal_viterbi(big_prob_map)
    
    grid_size_scaled = grid_size_original * scale_ratio
    print(f"   - حجم الشبكة الأصلي: {grid_size_original:.2f}")
    print(f"   - حجم الشبكة المعدل: {grid_size_scaled:.2f}")
    
    calibrated_signal = calibrate_signal(raw_signal, grid_size_scaled)
    
    plt.figure(figsize=(15, 8))
    
    plt.subplot(3, 1, 1)
    plt.title("Stitched Probability Map (Full Length)")
    plt.imshow(big_prob_map, cmap='magma', aspect='auto')
    plt.axis('off')
    
    plt.subplot(3, 1, 2)
    plt.title("Extracted Raw Signal")
    plt.plot(raw_signal)
    plt.xlim(0, len(raw_signal))
    plt.grid(True, alpha=0.3)
    
    plt.subplot(3, 1, 3)
    plt.title("Final Calibrated Signal (mV)")
    plt.plot(calibrated_signal, color='red', linewidth=1)
    plt.axhline(0, color='black', linewidth=0.5)
    plt.ylabel("mV")
    plt.xlim(0, len(calibrated_signal))
    plt.grid(True, which='both', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✅ هل ترى الآن موجات القلب (QRS) واضحة في الرسم الأحمر؟")

else:
    print("❌ لم يتم العثور على صور.")

In [ ]:
# خلية 14
def viterbi_extract_actual_signal(prob_map, lam=0.5, max_jump=100):
    H, W = prob_map.shape
    
    cost_map = -np.log(prob_map + 1e-6)
    dp = np.full((H, W), np.inf)
    parent = np.zeros((H, W), dtype=int)
    dp[:, 0] = cost_map[:, 0]

    for x in range(1, W):
        prev_col_costs = dp[:, x-1]
        for y in range(H):
            y_min = max(0, y - max_jump)
            y_max = min(H, y + max_jump + 1)
            
            window_costs = prev_col_costs[y_min:y_max]
            dist_penalty = lam * np.abs(np.arange(y_min, y_max) - y)
            total = window_costs + dist_penalty
            
            best_idx = np.argmin(total)
            dp[y, x] = cost_map[y, x] + total[best_idx]
            parent[y, x] = y_min + best_idx

    path = np.zeros(W, dtype=int)
    path[-1] = np.argmin(dp[:, -1])
    for x in range(W-2, -1, -1):
        path[x] = parent[path[x+1], x+1]
        
    final_signal = []
    for x, y_int in enumerate(path):
        y_start = max(0, y_int - 3)
        y_end = min(H, y_int + 4)
        weights = prob_map[y_start:y_end, x]
        
        if np.sum(weights) > 1e-5:
            y_sub = np.sum(weights * np.arange(y_start, y_end)) / np.sum(weights)
        else:
            y_sub = y_int
            
        final_signal.append(H - y_sub)
        
    return np.array(final_signal)

print("✅ تم تحديث دالة Viterbi لتخرج إشارة (Amplitude) مباشرة.")

In [ ]:
# --- الخلية 15: المحرك الهندسي المتقدم (Advanced Geometry Engine) ---

def correct_skew_fft(image):
    """
    تستخدم تحويل فورييه (FFT) لاكتشاف دوران الشبكة بدقة متناهية.
    """
    try:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        
        # 1. تطبيق FFT
        f = np.fft.fft2(gray)
        fshift = np.fft.fftshift(f)
        magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-6)

        # 2. حذف الترددات المنخفضة (المركز) للتركيز على الخطوط الدقيقة
        cy, cx = h // 2, w // 2
        r = 20
        magnitude_spectrum[cy-r:cy+r, cx-r:cx+r] = 0

        # 3. العثور على أكثر الزوايا تكراراً في الطيف
        # نستخدم HoughLines على طيف الترددات وليس الصورة الأصلية!
        # (هذا سر المهنة: الخطوط في الطيف تعامد الخطوط في الصورة)
        _, binary = cv2.threshold(magnitude_spectrum.astype(np.uint8), 150, 255, cv2.THRESH_BINARY)
        lines = cv2.HoughLines(binary, 1, np.pi / 180, 50)

        angles = []
        if lines is not None:
            for rho, theta in lines[:, 0]:
                angle = np.degrees(theta) - 90
                # نحن نبحث عن زوايا قريبة من 0 أو 90
                if -10 < angle < 10:
                    angles.append(angle)
                elif 80 < angle < 100:
                    angles.append(angle - 90)
        
        if not angles:
            return image, 0.0

        median_angle = np.median(angles)
        
        # تدوير الصورة
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, median_angle, 1.0)
        rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        
        return rotated, median_angle

    except Exception as e:
        # print(f"⚠️ FFT Skew Failed: {e}")
        return image, 0.0

def correct_skew_backup(image):
    """
    الطريقة التقليدية (Safety Backup) باستخدام MinAreaRect
    """
    if image is None: return image, 0.0
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.bitwise_not(gray)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    
    if len(coords) == 0: return image, 0.0

    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45: angle = -(90 + angle)
    else: angle = -angle
    
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    
    return rotated, angle

def fix_image_orientation(image):
    """
    المدير الذكي: يختار أفضل طريقة للتصحيح
    """
    if image is None: return None

    # 1. المحاولة الأولى: دقة عالية بـ FFT
    rotated_fft, angle_fft = correct_skew_fft(image)
    
    # إذا كانت الزاوية منطقية (أقل من 5 درجات)، نعتمدها
    if abs(angle_fft) > 0.1 and abs(angle_fft) < 5.0:
        # print(f"📐 تم التصحيح بدقة FFT: {angle_fft:.2f}°")
        return rotated_fft

    # 2. المحاولة الثانية: إذا فشل FFT أو كانت الصورة معقدة، نستخدم الطريقة التقليدية
    # لكن بحذر (Safety Lock)
    rotated_backup, angle_backup = correct_skew_backup(image)
    
    if abs(angle_backup) > 0.5 and abs(angle_backup) < 15.0: # القفل: لا تصحح أكثر من 15 درجة
        # print(f"📐 تم التصحيح بالطريقة التقليدية: {angle_backup:.2f}°")
        return rotated_backup
    
    # 3. إذا لم نجد ميلاً واضحاً، أعد الصورة كما هي
    return image

print("✅ تم تحديث المحرك الهندسي (FFT + Safety Lock).")

In [ ]:
# خلية 16
def preprocess_remove_grid(image):
    if image is None: return None
    
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    lower_red1 = np.array([0, 50, 50])
    upper_red1 = np.array([10, 255, 255])
    
    lower_red2 = np.array([170, 50, 50])
    upper_red2 = np.array([180, 255, 255])
    
    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    grid_mask = mask1 + mask2
    
    kernel = np.ones((2,2), np.uint8)
    grid_mask = cv2.dilate(grid_mask, kernel, iterations=1)
    
    image_clean = image.copy()
    image_clean[grid_mask > 0] = (255, 255, 255)
    
    return image_clean

In [ ]:
# --- الخلية 17: اختبار يدوي (مصحح) ---
def get_12_leads_crops(image):
    # دالة قص بسيطة للاختبار السريع (Blind Crop)
    h, w, c = image.shape
    margin_x = int(w * 0.05); margin_y = int(h * 0.05)
    active_area = image[margin_y:h-margin_y, margin_x:w-margin_x]
    ah, aw, _ = active_area.shape
    row_h = ah // 4; col_w = aw // 3
    crops = []
    for r in range(4):
        for c in range(3):
            crops.append(active_area[r*row_h:(r+1)*row_h, c*col_w:(c+1)*col_w])
    return crops[:12]

print("🕵️‍♂️ جاري جرد الصور المتاحة فعلياً...")
image_files = []
search_roots = ['/kaggle/input']
if 'dataset_path' in globals(): search_roots.append(dataset_path)

for root in search_roots:
    image_files += glob.glob(f"{root}/**/*.png", recursive=True)
    image_files += glob.glob(f"{root}/**/*.jpg", recursive=True)

print(f"✅ وجدنا {len(image_files)} صورة في المجلدات.")

if len(image_files) == 0:
    print("❌ كارثة: لا توجد أي صور! تأكد من إضافة البيانات (Add Data).")
else:
    sample_files = random.sample(image_files, min(3, len(image_files)))
    
    submission_rows = []
    print(f"🚀 بدء معالجة {len(sample_files)} صور عشوائية من الموجودة فعلاً...")

    for img_path in tqdm(sample_files):
        file_name = os.path.basename(img_path)
        img_id = os.path.splitext(file_name)[0]
        
        image = cv2.imread(img_path)
        leads_data = []
        
        if image is not None:
            try:
                # --- التعديل هنا: استخدام الدالة الجديدة من الخلية 15 ---
                image = fix_image_orientation(image) 
                if image is None: image = cv2.imread(img_path) # Fallback
                
                image = preprocess_remove_grid(image) 
                crops = get_12_leads_crops(image)
                
                for crop in crops:
                    # نستخدم grid_size افتراضي للاختبار السريع
                    grid_size = estimate_grid_size(crop, display=False)
                    if grid_size < 5: grid_size = 15.0 
                    
                    # نستخدم predict_long_image المعرفة في الخلية 13
                    prob_map, scale_ratio = predict_long_image(crop, model)
                    raw_sig = viterbi_extract_actual_signal(prob_map, lam=0.01, max_jump=100)                      
                    scaled_grid = grid_size * scale_ratio
                    calib_sig = calibrate_signal(raw_sig, scaled_grid)
                    
                    target_len = 1000 
                    resampled = resample(calib_sig, target_len)
                    
                    final_sig = -(resampled - np.mean(resampled))
                    leads_data.append(final_sig)
                    
            except Exception as e:
                print(f"⚠️ خطأ في معالجة {img_id}: {e}")
                leads_data = [np.zeros(1000) for _ in range(12)]
        else:
            leads_data = [np.zeros(1000) for _ in range(12)]

        if len(leads_data) < 12:
            leads_data = [np.zeros(1000) for _ in range(12)]

        row_dict = {'id': img_id}
        for i, sig in enumerate(leads_data):
            row_dict[f'lead_{i+1}'] = list(np.round(sig, 3))
        submission_rows.append(row_dict)

    final_df = pd.DataFrame(submission_rows)
    print("\n🎉 النتيجة النهائية (أرقام حقيقية):")
    
    if not final_df.empty:
        print(final_df.iloc[0]['lead_1'][:30]) 
    
    final_df.to_csv("submission_from_images.csv", index=False)
    print("💾 تم حفظ الملف: submission_from_images.csv")

In [ ]:
# --- الخلية 18: إعداد بيانات YOLO القتالية (Combat Mode) - [M4: UPDATED] ---

BASE_DIR = "yolo_dataset"
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)

os.makedirs(f"{BASE_DIR}/images/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/images/val", exist_ok=True)
os.makedirs(f"{BASE_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/labels/val", exist_ok=True)

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

class CombatYOLOGenerator:
    def __init__(self, renderer):
        self.renderer = renderer
        self.dpi = 100
        
        # مؤثرات بصرية قاسية (لا تغير أماكن الصناديق، فقط تشوه الصورة)
        self.visual_aug = A.Compose([
            A.RandomShadow(num_shadows_lower=1, num_shadows_upper=4, 
                           shadow_dimension=8, shadow_roi=(0, 0, 1, 1), p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.6),
            A.GaussNoise(var_limit=(20.0, 100.0), p=0.6),
            A.ImageCompression(quality_lower=20, quality_upper=70, p=0.5),
            A.CoarseDropout(max_holes=15, max_height=40, max_width=40, min_holes=5, p=0.4),
            A.ToGray(p=0.2), # أحياناً تكون الصورة أبيض وأسود تماماً
        ])
        
    def add_distractors(self, image):
        """إضافة خربشات وكتابة على الصفحة كاملة"""
        h, w = image.shape[:2]
        # كتابة عشوائية
        if random.random() > 0.3:
            texts = ["CONFIDENTIAL", "ECG REPORT", "PhysioNet", "Error: 404", "Leads V1-V6"]
            txt = random.choice(texts)
            font = cv2.FONT_HERSHEY_SIMPLEX
            # حجم ومكان عشوائي
            cv2.putText(image, txt, (random.randint(50, w-200), random.randint(50, h-50)), 
                        font, random.uniform(1.0, 3.0), (0,0,0), 3, cv2.LINE_AA)
            
        # خطوط عشوائية (قلم جاف)
        if random.random() > 0.4:
            for _ in range(random.randint(1, 3)):
                pts = np.array([[random.randint(0, w), random.randint(0, h)] for _ in range(4)], np.int32)
                pts = pts.reshape((-1, 1, 2))
                col = random.choice([(0,0,200), (50,50,50)])
                cv2.polylines(image, [pts], False, col, 2)
        return image

    def create_combat_page(self, filename, subset="train"):
        fig_w, fig_h = 12, 8
        fig = plt.figure(figsize=(fig_w, fig_h), dpi=self.dpi)
        
        # خلفيات متنوعة (ورق قديم، أزرق، أبيض)
        bg_color = random.choice(['white', '#fcfcfc', '#f0f0f0', '#fffbf0'])
        fig.patch.set_facecolor(bg_color)
        
        # شبكة متغيرة (أحياناً غير موجودة لمحاكاة المسح السيء)
        if random.random() > 0.1:
            grid_col = random.choice(['red', 'pink', 'gray', 'black'])
            plt.minorticks_on()
            plt.grid(which='major', linestyle='-', linewidth=0.6, color=grid_col, alpha=random.uniform(0.2, 0.6))
            plt.grid(which='minor', linestyle=':', linewidth=0.3, color=grid_col, alpha=random.uniform(0.1, 0.4))
        
        rows, cols = 4, 3
        labels_data = []
        lead_idx = 0
        
        for r in range(rows):
            for c in range(cols):
                if lead_idx >= 12: break
                
                ax = fig.add_subplot(rows, cols, lead_idx + 1)
                
                # سحب إشارة حقيقية
                sig = self.renderer.get_real_signal()
                # تقصير للطول المناسب للعرض
                if len(sig) > 300: sig = sig[:300]
                
                # رسم خط الإشارة (سماكة ولون متغير)
                line_w = random.uniform(0.8, 2.0)
                line_c = 'black' if random.random() > 0.1 else 'blue'
                ax.plot(sig, color=line_c, linewidth=line_w)
                
                # تسمية الـ Lead (موجودة أو مخفية)
                lead_name = LEAD_NAMES[lead_idx]
                if random.random() > 0.2:
                    ax.text(0, 1, lead_name, fontsize=random.randint(8, 14), 
                            color='black', fontweight='bold')
                
                ax.axis('off')
                
                # حساب الصندوق
                bbox = ax.get_position()
                x0, y0, w, h = bbox.x0, bbox.y0, bbox.width, bbox.height
                x_center = x0 + w / 2
                y_center = 1.0 - (y0 + h / 2)
                
                # توسيع الصندوق قليلاً لضمان عدم قص القمم
                labels_data.append(f"{lead_idx} {x_center:.6f} {y_center:.6f} {w*1.15:.6f} {h*1.15:.6f}")
                lead_idx += 1
        
        # تحويل الشكل لصورة OpenCV لتطبيق المؤثرات
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        plt.close(fig)
        
        # 1. إضافة العناصر المربكة
        img = self.add_distractors(img)
        
        # 2. تطبيق التشوه البصري (Albumentations)
        # ملاحظة: نطبق مؤثرات بصرية فقط لا تغير الإحداثيات، لذا الـ labels تبقى صحيحة
        img = self.visual_aug(image=img)['image']
        
        # الحفظ
        cv2.imwrite(f"{BASE_DIR}/images/{subset}/{filename}.jpg", img)
        
        label_path = f"{BASE_DIR}/labels/{subset}/{filename}.txt"
        with open(label_path, "w") as f:
            f.write("\n".join(labels_data))

# استخدام renderer من الخلية 3
combat_gen = CombatYOLOGenerator(renderer)

print("🏭 بدء إنتاج صور YOLO القتالية (Combat Mode)...")
# زيادة العدد قليلاً لأن الموديل أكبر (v8x) ويحتاج بيانات أكثر
for i in tqdm(range(200), desc="Train Data"): 
    combat_gen.create_combat_page(f"train_{i}", subset="train")

for i in tqdm(range(50), desc="Val Data"):
    combat_gen.create_combat_page(f"val_{i}", subset="val")

# ملف Config
yaml_content = f"""
path: {os.path.abspath(BASE_DIR)} 
train: images/train
val: images/val
nc: 12
names: {LEAD_NAMES}
"""
with open(f"{BASE_DIR}/data.yaml", "w") as f:
    f.write(yaml_content)

print("✅ البيانات القتالية جاهزة.")

In [ ]:
# --- الخلية 19: تدريب YOLOv8x القتالي [M4: COMBAT TRAINING] ---

DATA_YAML = os.path.abspath("yolo_dataset/data.yaml")

# أ. النموذج: استخدام YOLOv8x (Extra Large) لأقصى دقة
# ملاحظة: إذا واجهت مشكلة في ذاكرة GPU، استبدله بـ 'yolov8l.pt' أو 'yolov8m.pt'
print("🦕 تحميل الوحش YOLOv8x...")
model_yolo = YOLO("yolov8x.pt") 

print("🚀 بدء التدريب القتالي (Mosaic=1.0, Degrees=15)...")

results = model_yolo.train(
    data=DATA_YAML,
    epochs=15, 
    imgsz=640,
    batch=8, # تقليل الباتش قليلاً لأن الموديل X ضخم
    name="ecg_combat_detector",
    
    # ب. إعدادات التدريب القتالية
    mosaic=1.0,   # تفعيل الموزاييك بقوة (دمج 4 صور) لتعقيد المشهد
    degrees=15.0, # تدوير الصور +/- 15 درجة لتعلم الصور المائلة
    shear=2.5,    # قص مائل بسيط
    perspective=0.0005, # تغيير المنظور
    
    verbose=True
)

print("✅ انتهى التدريب القتالي!")
best_path = "runs/detect/ecg_combat_detector/weights/best.pt"
if os.path.exists(best_path):
    print(f"💾 تم حفظ النموذج القتالي في: {best_path}")

In [ ]:
# --- الخلية 20: مصنع البيانات المصفح والتدريب التكميلي (Phase 2: Hardening) ---

# --- 1. إعداد المسارات ---
TRAIN_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/train.csv"
DATA_DIR = "/kaggle/input/physionet-ecg-image-digitization/train"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if os.path.exists(TRAIN_CSV_PATH):
    df_train = pd.read_csv(TRAIN_CSV_PATH)
    print(f"✅ تم تحميل {len(df_train)} ملف إشارة للتدريب.")
else:
    print("⚠️ ملف train.csv غير موجود!")
    df_train = pd.DataFrame()

# --- 2. تعريف التعزيزات القاسية (Hard Augmentations) ---
def get_heavy_augmentations():
    return A.Compose([
        # تشوهات هندسية (ورقة مكرمشة)
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03, p=0.3),
        A.GridDistortion(p=0.3),
        A.Rotate(limit=10, p=0.5), # دوران أكبر قليلاً
        
        # مشاكل الإضاءة والظلال (تصوير موبايل)
        A.RandomShadow(num_shadows_lower=1, num_shadows_upper=3, 
                       shadow_dimension=5, shadow_roi=(0, 0.5, 1, 1), p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
        
        # مشاكل الطباعة والورق
        A.CoarseDropout(max_holes=10, max_height=20, max_width=20, p=0.3), # بقع حبر أو ثقوب
        A.GaussianBlur(blur_limit=(3, 7), p=0.3), # صورة غير مركزة
        A.ImageCompression(quality_range=(30, 80), p=0.4), # جودة صورة سيئة
        A.GaussNoise(var_limit=(50.0, 150.0), p=0.4), # ضجيج عالي (ISO عالي)
        
        # تغييرات لونية
        A.RGBShift(r_shift_limit=30, g_shift_limit=30, b_shift_limit=30, p=0.3),
        A.ToGray(p=0.1), # أحياناً تكون الصورة أبيض وأسود
    ])

# --- 3. مصنع البيانات (RealSignalDataset) ---
class RealSignalDataset(Dataset):
    def __init__(self, dataframe, root_dir, augment=True):
        self.df = dataframe
        self.root_dir = root_dir
        self.augment = augment
        self.leads = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
        self.aug = get_heavy_augmentations() # استخدام القائمة القاسية

    def __len__(self):
        return len(self.df)

    def generate_image_pair(self, signal):
        # زيادة الدقة الأساسية قليلاً لتتحمل التشويه
        fig_h, fig_w = 3.0, 6.0; dpi = 120 
        
        # Mask (الهدف النظيف)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0); ax.axis('off') # خط أعرض قليلاً
        ax.set_ylim(np.min(signal)-0.5, np.max(signal)+0.5); fig.patch.set_facecolor('black')
        fig.canvas.draw()
        mask_img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask_img = mask_img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask_img = cv2.cvtColor(mask_img, cv2.COLOR_RGB2GRAY)
        _, mask_img = cv2.threshold(mask_img, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # Image (الدخل المشوه)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        # خلفيات ملونة عشوائية (وردي، أبيض، أصفر قديم)
        bg_color = random.choice(['white', '#fff5e6', '#ffe6e6', '#f0f0f0']) 
        grid_color = random.choice(['red', 'pink', 'black', 'grey', '#ffb6c1'])
        
        ax.set_facecolor(bg_color)
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=0.7)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=0.4)
        
        # رسم نصوص عشوائية (محاكاة ملاحظات الطبيب)
        if random.random() > 0.6:
            ax.text(random.uniform(0, len(signal)), np.max(signal), "V1", fontsize=15, color='black', alpha=0.7)
            
        ax.plot(signal, color='black', linewidth=1.3); ax.axis('off')
        ax.set_ylim(np.min(signal)-0.5, np.max(signal)+0.5)
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        input_img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        input_img = input_img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        plt.close(fig)
        
        return input_img, mask_img

    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]; rec_id = str(row['id'])
            sig_path = os.path.join(self.root_dir, rec_id, f"{rec_id}.csv")
            if not os.path.exists(sig_path): sig_path = glob.glob(os.path.join(self.root_dir, rec_id, "*.csv"))[0]
            
            sig_df = pd.read_csv(sig_path)
            lead = random.choice(self.leads); full_signal = sig_df[lead].values
            segment_len = random.randint(600, 1200)
            start = random.randint(0, max(0, len(full_signal) - segment_len))
            signal_segment = full_signal[start : start+segment_len]
            if np.isnan(signal_segment).any(): signal_segment = np.nan_to_num(signal_segment)

            image, mask = self.generate_image_pair(signal_segment)
            
            if self.augment:
                aug = self.aug(image=image, mask=mask)
                image, mask = aug['image'], aug['mask']
            
            image = cv2.resize(image, (512, 256))
            mask = cv2.resize(mask, (512, 256), interpolation=cv2.INTER_NEAREST)
            
            image = image.transpose(2, 0, 1).astype('float32') / 255.0
            mask = mask.astype('float32') / 255.0
            mask = np.expand_dims(mask, axis=0)
            
            return torch.from_numpy(image), torch.from_numpy(mask)
        except:
            # Fallback
            t = np.linspace(0, 10, 500); fake = np.sin(t)
            img, msk = self.generate_image_pair(fake)
            img = cv2.resize(img, (512, 256)).transpose(2,0,1).astype('float32')/255.0
            msk = cv2.resize(msk, (512, 256)).astype('float32')/255.0
            return torch.from_numpy(img), torch.from_numpy(np.expand_dims(msk, 0))

# --- 4. التدريب التكميلي (Fine-Tuning Loop) ---
BATCH_SIZE = 16
EPOCHS = 5 # عدد دورات قليل لأننا نكمل تدريب
LR = 0.0001 # معدل تعلم منخفض جداً للحفاظ على المعلومات السابقة

full_dataset = RealSignalDataset(df_train, DATA_DIR)
train_size = int(0.9 * len(full_dataset)) # نزيد نسبة التدريب
val_size = len(full_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# تحميل الموديل السابق كنقطة انطلاق
model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1)

PREV_MODEL_PATH = "best_model_real_data.pth"
if os.path.exists(PREV_MODEL_PATH):
    print(f"🔄 جاري تحميل الموديل السابق: {PREV_MODEL_PATH} لإكمال التدريب...")
    model.load_state_dict(torch.load(PREV_MODEL_PATH, map_location=DEVICE))
else:
    print("⚠️ لم يتم العثور على موديل سابق، سيتم التدريب من الصفر (قد يأخذ وقتاً أطول).")

model.to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = smp.losses.DiceLoss(mode='binary')

best_val_loss = float('inf')
NEW_MODEL_NAME = "best_model_phase2.pth"

print("🔥 بدء مرحلة Hardening (بيانات قبيحة + Learning Rate منخفض)...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    loop = tqdm(train_loader, desc=f"Phase 2 Epoch {epoch+1}/{EPOCHS}")
    
    for images, masks in loop:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            val_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1} -> Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")
    
    # Save Condition
    if avg_val_loss < best_val_loss:
        print(f"💎 تحسن في التعامل مع البيانات الصعبة! الحفظ في {NEW_MODEL_NAME}...")
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), NEW_MODEL_NAME)

print(f"✅ انتهت المرحلة الثانية. الموديل الجديد جاهز: {NEW_MODEL_NAME}")

In [ ]:
# خلية 21 
def correct_skew_orientation(image):
    if image is None: return None
    
    # تحويل للرمادي وحساب الزاوية
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.bitwise_not(gray)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    angle = cv2.minAreaRect(coords)[-1]
    
    # ضبط الزاوية
    if angle < -45: angle = -(90 + angle)
    else: angle = -angle
    
    # --- التعديل الجديد: قفل الأمان (Safety Lock) ---
    # إذا كانت الزاوية أكبر من 15 درجة، فهذا غالباً خطأ في اكتشاف الخطوط العمودية للشبكة
    # بدلاً من تدمير الصورة بتدويرها 90 درجة، نتجاهل التصحيح.
    if abs(angle) > 15.0:
        # print(f"⚠️ تم تجاهل دوران بزاوية {angle:.2f} (خطرة جداً).")
        return image

    if abs(angle) < 0.5: return image # زاوية صغيرة جداً لا تحتاج تصحيح

    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated

def get_yolo_crops(image, model):
    if model is None: return None
    try:
        # التوقع
        results = model.predict(image, verbose=False, conf=0.25)
        boxes = results[0].boxes.cpu().numpy()
        
        found = {}
        for box in boxes:
            cls_id = int(box.cls[0]); conf = box.conf[0]; xyxy = box.xyxy[0]
            # نحتفظ بأفضل صندوق لكل Lead
            if cls_id in found:
                if conf > found[cls_id]['conf']: found[cls_id] = {'box': xyxy, 'conf': conf}
            else: found[cls_id] = {'box': xyxy, 'conf': conf}
        
        # --- التعديل الجديد: سياسة الكل أو لا شيء (All or Nothing) ---
        # إذا وجد YOLO أقل من 12 إشارة، نعتبره فشل ونعود للقص التقليدي
        # هذا أفضل من إرسال أصفار أو إشارات خاطئة
        if len(found) < 12:
            return None 
            
        ordered_crops = []
        h_img, w_img = image.shape[:2]
        for i in range(12):
            # الآن نحن متأكدون أن المفتاح i موجود لأن len(found) == 12
            x1, y1, x2, y2 = map(int, found[i]['box'])
            x1, y1 = max(0, x1), max(0, y1); x2, y2 = min(w_img, x2), min(h_img, y2)
            ordered_crops.append(image[y1:y2, x1:x2])
            
        return ordered_crops
    except:
        return None

print("✅ تم تحديث الدوال الهندسية (Skew Safety + YOLO Fallback).")

In [18]:
# --- الخلية 22: المحرك البلاتيني (Phase 14: Signal Processing + Phase 15: TTA) - [FINAL BUILD] ---
# --- 1. إعداد المسارات ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

# --- 2. تحميل النماذج ---
print("⚙️ جاري تحميل المحرك البلاتيني (Phase 14+15)...")

YOLO_MODEL_PATH = "/kaggle/input/my-yolo-model/best.pt"
if not os.path.exists(YOLO_MODEL_PATH) and os.path.exists("best.pt"): YOLO_MODEL_PATH = "best.pt"

if os.path.exists(YOLO_MODEL_PATH):
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print(f"✅ YOLO (Combat): مفعل.")
else:
    print("⚠️ تحذير: YOLO غير موجود.")
    yolo_model = None

UNET_MODEL_PATH = "best_model_phase10.pth" 
unet_model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=1, decoder_attention_type="scse")

if os.path.exists(UNET_MODEL_PATH):
    unet_model.load_state_dict(torch.load(UNET_MODEL_PATH, map_location=DEVICE))
    print("💎 U-Net (Phase 10): مفعل.")
else:
    print(f"⚠️ خطأ: لم يتم العثور على {UNET_MODEL_PATH}!")

unet_model.to(DEVICE)
unet_model.eval()

# --- 3. دوال المعالجة الطبية (Phase 14) ---

def apply_high_pass_filter(data, cutoff=0.5, fs=500, order=5):
    """
    Phase 14-B: إزالة انحراف الخط الأساسي (Baseline Wander)
    تستخدم فلتر بتروورث (Butterworth) لتمرير الترددات الأعلى من 0.5 هرتز فقط.
    """
    if len(data) < order * 3: return data # لا يمكن الفلترة إذا البيانات قليلة
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    # filtfilt يطبق الفلتر ذهاباً وإياباً لمنع انزياح الطور (Zero-phase)
    return filtfilt(b, a, data)

def smart_einthoven_fix(leads):
    """
    Phase 14-C: قانون أينتهوفن الذكي (Einthoven's Law)
    القانون: Lead II = Lead I + Lead III
    نقوم بتوزيع الخطأ بالتساوي على القنوات الثلاث لتقليل التشوه.
    """
    if 'I' in leads and 'II' in leads and 'III' in leads:
        l = min(len(leads['I']), len(leads['II']), len(leads['III']))
        I = leads['I'][:l]
        II = leads['II'][:l]
        III = leads['III'][:l]
        
        # حساب الخطأ المتبقي
        residual = (I + III) - II
        
        # توزيع الخطأ لتقليله (أفضل رياضياً من استبدال قناة واحدة)
        # II_new = II + error/3
        # I_new  = I  - error/3
        # III_new= III- error/3
        # هذا يضمن تحقق المعادلة بأقل تغيير في شكل الموجة الأصلي
        leads['II'][:l] += residual / 3.0
        leads['I'][:l]  -= residual / 3.0
        leads['III'][:l]-= residual / 3.0
        
    return leads

# --- 4. دوال التنبؤ المتقدمة (Phase 15: TTA) ---

def predict_with_tta(image, model):
    """
    Phase 15-B: Test Time Augmentation (TTA)
    نقوم بالتنبؤ مرتين:
    1. الصورة الأصلية.
    2. الصورة معكوسة أفقياً (Horizontal Flip).
    ثم ندمج النتيجتين. هذا يزيل التحيزات الاتجاهية للموديل.
    """
    h, w = image.shape[:2]; target_h = 256; scale = target_h / h; new_w = int(w * scale)
    img_rz = cv2.resize(image, (new_w, target_h))
    
    # تحضير الصورة الأصلية
    inp = torch.from_numpy(img_rz).permute(2,0,1).float().unsqueeze(0).to(DEVICE) / 255.0
    
    # تحضير الصورة المعكوسة (Flip)
    img_flip = cv2.flip(img_rz, 1) # 1 means horizontal flip
    inp_flip = torch.from_numpy(img_flip).permute(2,0,1).float().unsqueeze(0).to(DEVICE) / 255.0
    
    # التنبؤ (دفعة واحدة للسرعة إذا أمكن، لكن منفصلة أضمن للذاكرة)
    with torch.no_grad():
        # التنبؤ الأصلي
        prob_orig = torch.sigmoid(model(inp)).cpu().numpy()[0][0]
        
        # التنبؤ المعكوس
        prob_flip = torch.sigmoid(model(inp_flip)).cpu().numpy()[0][0]
        
    # عكس نتيجة الفليب لتعود لطبيعتها
    prob_flip_back = cv2.flip(prob_flip, 1)
    
    # دمج النتيجتين (Average Ensemble)
    final_prob = (prob_orig + prob_flip_back) / 2.0
    
    # تطبيق قمع الـ Artifacts على النتيجة المدمجة
    final_prob = suppress_vertical_artifacts(final_prob)
    
    return final_prob, scale

# --- (باقي الدوال المساعدة كما هي من المرحلة السابقة) ---
def suppress_vertical_artifacts(prob_map, threshold_factor=2.5):
    col_means = np.mean(prob_map, axis=0)
    global_median = np.median(col_means)
    mask = np.ones_like(col_means)
    artifact_indices = col_means > (global_median * threshold_factor + 0.05)
    mask[artifact_indices] = 0.1
    return prob_map * mask[None, :]

def adaptive_viterbi(prob_map): # (نفس كود المرحلة 12)
    H, W = prob_map.shape
    snr = np.percentile(np.max(prob_map, axis=0), 95) / (np.percentile(np.max(prob_map, axis=0), 10) + 1e-6)
    lam = 0.05 if snr > 5 else (0.5 if snr > 2 else 2.0)
    dp = np.full((H, W), np.inf); par = np.zeros((H, W), int)
    dp[:, 0] = -np.log(prob_map[:, 0] + 1e-6)
    for x in range(1, W):
        prev_col = dp[:, x-1]
        current_probs = -np.log(prob_map[:, x] + 1e-6)
        for y in range(H):
            window_size = 5 if snr > 3 else 3
            y_start = max(0, y - window_size); y_end = min(H, y + window_size + 1)
            candidates = prev_col[y_start:y_end] + lam * np.abs(np.arange(y_start, y_end) - y)
            best_local_idx = np.argmin(candidates)
            dp[y, x] = candidates[best_local_idx] + current_probs[y]
            par[y, x] = y_start + best_local_idx
    path = np.zeros(W, int); path[-1] = np.argmin(dp[:, -1])
    for x in range(W - 2, -1, -1): path[x] = par[path[x + 1], x + 1]
    final_path = []
    for x, y in enumerate(path):
        y_start, y_end = max(0, y - 2), min(H, y + 3)
        weights = prob_map[y_start:y_end, x]
        if np.sum(weights) > 1e-5: final_path.append(H - (np.sum(weights * np.arange(y_start, y_end)) / np.sum(weights)))
        else: final_path.append(H - y)
    return np.array(final_path)

def robust_multi_point_calibration(crop, default_val=25.0): # (نفس كود المرحلة 13)
    if crop is None or crop.size == 0: return default_val
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)); gray = clahe.apply(gray)
    edges_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    edges_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    peaks_x_indices, _ = find_peaks(np.sum(np.abs(edges_x), axis=0), distance=10, prominence=50)
    peaks_y_indices, _ = find_peaks(np.sum(np.abs(edges_y), axis=1), distance=10, prominence=50)
    grid_sizes = []
    if len(peaks_x_indices) > 3: grid_sizes.extend(np.diff(peaks_x_indices)[(np.diff(peaks_x_indices)>10) & (np.diff(peaks_x_indices)<100)])
    if len(peaks_y_indices) > 3: grid_sizes.extend(np.diff(peaks_y_indices)[(np.diff(peaks_y_indices)>10) & (np.diff(peaks_y_indices)<100)])
    if len(grid_sizes) >= 5:
        grid_sizes = np.array(grid_sizes)
        q1 = np.percentile(grid_sizes, 25); q3 = np.percentile(grid_sizes, 75); iqr = q3 - q1
        clean_grid = grid_sizes[(grid_sizes >= q1 - 1.5*iqr) & (grid_sizes <= q3 + 1.5*iqr)]
        if len(clean_grid) > 0: return (0.6 * np.median(clean_grid)) + (0.4 * np.mean(clean_grid))
    return default_val

def advanced_perspective_correction(image): # (كما هي)
    if image is None: return None
    h, w = image.shape[:2]
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=w//4, maxLineGap=20)
    if lines is None: return image
    horizontal_points = []
    for line in lines:
        x1, y1, x2, y2 = line[0]; angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
        if abs(angle) < 10: horizontal_points.append([x1, y1]); horizontal_points.append([x2, y2])
    if len(horizontal_points) < 10: return image
    horizontal_points = np.array(horizontal_points)
    try:
        from skimage.measure import LineModelND
        model, inliers = ransac(horizontal_points, LineModelND, min_samples=2, residual_threshold=5, max_trials=100)
        vector = model.params[1]; angle_deg = np.degrees(np.arctan2(vector[1], vector[0]))
        if abs(angle_deg) > 0.5 and abs(angle_deg) < 15:
            center = (w // 2, h // 2); M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
            return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    except: pass
    return image

def get_yolo_crops_with_fallback(image, model): # (كما هي)
    if model:
        try:
            results = model.predict(image, verbose=False, conf=0.15, imgsz=640)
            found = {}
            for box in results[0].boxes.cpu().numpy():
                cid = int(box.cls[0]); conf = box.conf[0]
                if cid not in found or conf > found[cid]['conf']: found[cid] = {'box': box.xyxy[0], 'conf': conf}
            if len(found) == 12:
                crops = []; h, w = image.shape[:2]
                for i in range(12):
                    x1,y1,x2,y2 = map(int, found[i]['box']); pad=5
                    crops.append(image[max(0,y1-pad):min(h,y2+pad), max(0,x1-pad):min(w,x2+pad)])
                return crops
        except: pass
    h, w = image.shape[:2]; rh = h//4; cw = w//3
    return [image[r*rh:(r+1)*rh, c*cw:(c+1)*cw] for r in range(4) for c in range(3)]

def preprocess_remove_grid_rgb(image_rgb): # (كما هي)
    if image_rgb is None: return None
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV) 
    mask = cv2.inRange(hsv, np.array([0, 50, 50]), np.array([10, 255, 255])) + \
           cv2.inRange(hsv, np.array([170, 50, 50]), np.array([180, 255, 255]))
    mask = cv2.dilate(mask, np.ones((2,2), np.uint8), iterations=1)
    img_c = image_rgb.copy(); img_c[mask > 0] = (255, 255, 255)
    return img_c

# --- 5. التنفيذ النهائي ---
if os.path.exists(TEST_CSV_PATH): test_df = pd.read_csv(TEST_CSV_PATH)
else: test_df = pd.DataFrame({'id': ['001'], 'fs': [500], 'number_of_rows': [5000]})
paths = glob.glob(f"{IMAGE_DIR}/**/*.png") + glob.glob(f"{IMAGE_DIR}/**/*.jpg")
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in paths}
ids_list, vals_list = [], []

print(f"🚀 بدء الاستخراج البلاتيني (Phase 14+15: TTA & Medical Polish) لـ {len(test_df)} ملف...")

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    sid = str(row['id'])
    tlen = int(row['number_of_rows']) if 'number_of_rows' in row and not pd.isna(row['number_of_rows']) else int(10*(row.get('fs',500)))
    fs = float(row['fs']) if 'fs' in row and not pd.isna(row['fs']) else 500.0 # قراءة التردد للفلترة
    path = path_map.get(sid)
    leads = {}
    if path:
        try:
            img = cv2.imread(path); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = advanced_perspective_correction(img)
            global_grid = robust_multi_point_calibration(img, 25.0) 
            crops = get_yolo_crops_with_fallback(img, yolo_model)
            crops = [preprocess_remove_grid_rgb(c) for c in crops]
            for i, crop in enumerate(crops):
                lname = LEAD_NAMES[i]
                if crop.size < 100: leads[lname] = np.zeros(tlen); continue
                
                local_grid = robust_multi_point_calibration(crop, default_val=global_grid)
                
                # [Phase 15] استخدام TTA بدلاً من التنبؤ العادي
                prob, scale = predict_with_tta(crop, unet_model)
                
                raw = adaptive_viterbi(prob)
                g_sc = local_grid * scale
                sig_mv = (raw - np.median(raw)) / (g_sc * 10 if g_sc > 1 else 250)
                
                # [Phase 14-A] التنعيم
                if len(sig_mv)>15: sig_mv = savgol_filter(sig_mv, 11, 3)
                
                # [Phase 14-B] فلتر High-pass لإزالة انحراف الأساس
                if len(sig_mv) > 20: sig_mv = apply_high_pass_filter(sig_mv, cutoff=0.5, fs=fs)
                
                leads[lname] = resample(sig_mv, tlen) if len(sig_mv)>0 else np.zeros(tlen)
            
            # [Phase 14-C] تصحيح أينتهوفن
            leads = smart_einthoven_fix(leads)
            
        except: leads = {l: np.zeros(tlen) for l in LEAD_NAMES}
    else: leads = {l: np.zeros(tlen) for l in LEAD_NAMES}
    for l in LEAD_NAMES:
        s = leads.get(l, np.zeros(tlen))
        if len(s)!=tlen: s = resample(s, tlen) if len(s)>0 else np.zeros(tlen)
        ids_list.extend([f"{sid}_{i}_{l}" for i in range(tlen)])
        vals_list.extend(s)

print("💾 الحفظ النهائي (Platinum Edition)...")
pd.DataFrame({'id': ids_list, 'value': vals_list}).to_parquet("submission.parquet", index=False)
print("🏆 تمت المهمة! هذا أقوى ما يمكن أن تنتجه التكنولوجيا الحالية.")

⚙️ جاري تحميل المحرك البلاتيني (Phase 14+15)...
✅ YOLO (Combat): مفعل.
💎 U-Net (Phase 10): مفعل.
🚀 بدء الاستخراج البلاتيني (Phase 14+15: TTA & Medical Polish) لـ 24 ملف...


100%|██████████| 24/24 [04:41<00:00, 11.72s/it]


💾 الحفظ النهائي (Platinum Edition)...
🏆 تمت المهمة! هذا أقوى ما يمكن أن تنتجه التكنولوجيا الحالية.


In [20]:
# خلية 23
MY_SUB_PATH = "submission.parquet"
OFFICIAL_SAMPLE_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"

print("🔍 جاري التحقق من التنسيق الطويل (Long Format)...")

try:
    my_sub = pd.read_parquet(MY_SUB_PATH)
    try:
        sample_sub = pd.read_parquet(OFFICIAL_SAMPLE_PATH)
        has_sample = True
    except:
        has_sample = False
        print("⚠️ لم نتمكن من قراءة sample_submission، سنفحص الهيكل فقط.")

    required_columns = {'id', 'value'}
    my_columns = set(my_sub.columns)
    
    if my_columns != required_columns:
        print(f"❌ خطأ قاتل: الأعمدة غير صحيحة!")
        print(f"الموجود لديك: {my_columns}")
        print(f"المطلوب: {required_columns}")
        raise ValueError("Column Mismatch")
    else:
        print("✅ أسماء الأعمدة صحيحة (id, value).")

    first_value = my_sub.iloc[0]['value']
    if not isinstance(first_value, (float, np.floating)):
        print(f"⚠️ تحذير: نوع القيم قد لا يكون float. النوع المكتشف: {type(first_value)}")
        print("جاري محاولة التحويل للتأكد...")
        try:
            float(first_value)
            print("✅ القيم قابلة للتحويل لأرقام.")
        except:
            print("❌ القيم ليست أرقاماً!")
            raise ValueError("Value Type Error")
    else:
        print("✅ نوع البيانات (float) صحيح.")

    first_id = my_sub.iloc[0]['id']
    print(f"🔍 عينة من الـ ID في ملفك: {first_id}")
    if "_" not in str(first_id):
        print("⚠️ تحذير: شكل الـ ID يبدو غريباً (لا يحتوي على underscore).")
    
    print(f"📊 إجمالي عدد الصفوف في ملفك: {len(my_sub)}")
    
    if has_sample:
        print(f"📊 إجمالي عدد الصفوف في Sample: {len(sample_sub)}")
        
        if len(my_sub) == len(sample_sub):
            print("✅ تطابق تام في عدد الصفوف!")
        else:
            print("ℹ️ عدد الصفوف مختلف (وهذا طبيعي أحياناً إذا كان test.csv مختلف عن sample).")
            if len(my_sub) % 12 == 0:
                 print("✅ عدد الصفوف يقبل القسمة على 12 (منطقي لـ 12 leads).")
            else:
                 print("⚠️ تحذير: عدد الصفوف لا يقبل القسمة على 12!")

    print("\n🎉 النتيجة النهائية: هيكل الملف سليم 100% للتنسيق الطويل.")
    print("توكل على الله وارفع الملف (Submit)!")

except Exception as e:
    print(f"\n❌ حدث خطأ أثناء التحقق: {e}")

🔍 جاري التحقق من التنسيق الطويل (Long Format)...
✅ أسماء الأعمدة صحيحة (id, value).
✅ نوع البيانات (float) صحيح.
🔍 عينة من الـ ID في ملفك: 1053922973_0_I
📊 إجمالي عدد الصفوف في ملفك: 900000
📊 إجمالي عدد الصفوف في Sample: 75000
ℹ️ عدد الصفوف مختلف (وهذا طبيعي أحياناً إذا كان test.csv مختلف عن sample).
✅ عدد الصفوف يقبل القسمة على 12 (منطقي لـ 12 leads).

🎉 النتيجة النهائية: هيكل الملف سليم 100% للتنسيق الطويل.
توكل على الله وارفع الملف (Submit)!
